# Worktrees: zwei Arbeitsordner, ein Repo

Dasselbe wie in der Live-Demo mit AstroPlanner, hier am Repo mit den Workshop-Notebooks. Ein Worktree ist ein zweiter Arbeitsordner mit eigenem Branch. Historie und Objekte liegen im gemeinsamen `.git`, getrennt sind nur die Dateien. So bekommt jeder Agent seine eigene Kopie.

In fünf Schritten, alles mit Git in `%%bash`-Zellen:

1. Repo klonen
2. Zwei Worktrees anlegen
3. In einem Worktree eine Datei ändern
4. Denselben Branch ein zweites Mal auschecken
5. Aufräumen

### Pflicht (rund 7 Minuten)

Die Schritte 1 bis 5 der Reihe nach ausführen und die Ausgaben lesen. Bei Schritt 4: Warum lässt Git das nicht zu?

### Zusatz (für Schnelle)

Vor dem Aufräumen: die Änderung aus Schritt 3 committen, vom anderen Worktree aus ansehen und dort mergen.

## 1. Repo klonen

Die erste Zelle legt fest, wo geübt wird: in Colab unter `/content/uebung-worktrees`, lokal in einem temporären Ordner. Die `%%bash`-Zellen danach finden den Ordner über die Variable `UEBUNG`.

In [1]:
import os
import tempfile
from pathlib import Path

try:
    import google.colab  # noqa: F401  gibt es nur in Colab
    UEBUNG = Path("/content/uebung-worktrees")
except ImportError:
    UEBUNG = Path(tempfile.gettempdir()) / "uebung-worktrees"
os.environ["UEBUNG"] = str(UEBUNG)
print(UEBUNG)

/var/folders/n0/3rmmylfj6xg_f6lfznjt8t380000gn/T/uebung-worktrees


Der Klon bekommt eine Git-Identität für die Commits im Zusatz. `git config` ohne `--global` gilt nur für diesen Klon und seine Worktrees.

In [2]:
%%bash
rm -rf "${UEBUNG:?zuerst die Zelle oben ausführen}" && mkdir -p "$UEBUNG" && cd "$UEBUNG"
git clone --quiet https://github.com/dprinz/ki-workshop-handson handson
cd handson
git config user.name "Workshop"
git config user.email "workshop@example.invalid"
git log -1 --oneline
ls

31838f4 README: in Colab gemessene Laufzeiten, Warnung beim ersten Ausführen, Sitzungslimit im Grat

is-Konto


README.md
birds
einstellungen-jupyterlab
overfitting


## 2. Zwei Worktrees anlegen

`git worktree add` legt neben dem Klon einen Ordner an und checkt dort einen neuen Branch aus, wie auf der Folie: `git worktree add ../repo-a -b feature-a`. Hier heissen die Branches wie in der Live-Demo `demo/a` und `demo/b`. Git schreibt seine Meldungen auf stderr; in Colab stehen sie deshalb manchmal farbig oder vor der übrigen Ausgabe.

In [3]:
%%bash
cd "$UEBUNG/handson"
git worktree add ../handson-a -b demo/a
git worktree add ../handson-b -b demo/b
ls ..
git worktree list

Bereite Arbeitsverzeichnis vor (neuer Branch 'demo/a')


HEAD ist jetzt bei 31838f4 README: in Colab gemessene Laufzeiten, Warnung beim ersten Ausführen, Si

tzungslimit im Gratis-Konto


Bereite Arbeitsverzeichnis vor (neuer Branch 'demo/b')


HEAD ist jetzt bei 31838f4 README: in Colab gemessene Laufzeiten, Warnung beim ersten Ausführen, Si

tzungslimit im Gratis-Konto


handson
handson-a
handson-b


/private/var/folders/n0/3rmmylfj6xg_f6lfznjt8t380000gn/T/uebung-worktrees/handson   31838f4 [main]
/

private/var/folders/n0/3rmmylfj6xg_f6lfznjt8t380000gn/T/uebung-worktrees/handson-a 31838f4 [demo/a]


/private/var/folders/n0/3rmmylfj6xg_f6lfznjt8t380000gn/T/uebung-worktrees/handson-b 31838f4 [demo/b]

Drei Zeilen in der Liste: der Klon auf `main` und die zwei Worktrees, alle auf demselben Commit. Neu entstanden sind nur die Dateien. Im Worktree ist `.git` kein Ordner, sondern eine Datei mit einem Verweis auf das gemeinsame `.git` des Klons:

In [4]:
%%bash
cd "$UEBUNG"
cat handson-a/.git
ls handson/.git/worktrees

gitdir: /private/var/folders/n0/3rmmylfj6xg_f6lfznjt8t380000gn/T/uebung-worktrees/handson/.git/workt

rees/handson-a


handson-a
handson-b


## 3. In einem Worktree eine Datei ändern

Im Worktree `handson-a` bekommt `README.md` eine Zeile dazu. Der andere Worktree sieht davon nichts.

In [5]:
%%bash
cd "$UEBUNG/handson-a"
echo "Geändert im Worktree handson-a" >> README.md

echo "handson-a, letzte Zeile der README:"
tail -1 README.md
git status --short

echo
echo "handson-b, letzte Zeile der README:"
tail -1 ../handson-b/README.md
git -C ../handson-b status --short | grep . || echo "git status: keine Änderung"

handson-a, letzte Zeile der README:


Geändert im Worktree handson-a


 M README.md


handson-b, letzte Zeile der README:


  von Jeremy Howard.


git status: keine Änderung


Zwei Agents in zwei Worktrees schreiben sich also nicht gegenseitig in die Dateien. Zusammen kommen die Änderungen erst beim Merge.

## 4. Denselben Branch ein zweites Mal auschecken

Ein dritter Worktree soll `demo/a` auschecken, den Branch, der schon in `handson-a` ausgecheckt ist.

In [6]:
%%bash
cd "$UEBUNG/handson"
git worktree add ../handson-c demo/a || echo "Exit-Code $?, kein neuer Worktree"

Bereite Arbeitsverzeichnis vor (checke 'demo/a' aus)


Schwerwiegend: 'demo/a' wird bereits von Arbeitsverzeichnis in '/private/var/folders/n0/3rmmylfj6xg_

f6lfznjt8t380000gn/T/uebung-worktrees/handson-a' verwendet


Exit-Code 128, kein neuer Worktree


Git bricht ab. Je nach Version lautet die Meldung „is already used by worktree at …“ oder „is already checked out at …“, auf einem deutschen System „wird bereits von Arbeitsverzeichnis in … verwendet“ wie in der Live-Demo.

**Frage:** Warum lässt Git das nicht zu?

In [7]:
# @title Lösung (aufklappen)
from IPython.display import Markdown

Markdown("""Ein Branch ist ein Zeiger auf einen Commit. Ein Commit in einem Worktree schiebt den Zeiger seines Branches weiter.
Wären zwei Worktrees auf demselben Branch, würde ein Commit im einen den Branch auch für den anderen weiterschieben.
Dessen Dateien passten dann nicht mehr zu seinem `HEAD`, und `git status` zeigte Änderungen, die dort niemand gemacht hat.

Deshalb gehört ein Branch zu genau einem Worktree, und jeder Agent bekommt seinen eigenen.
Mit `--force` geht es trotzdem; dann muss man selbst aufpassen.""")

Ein Branch ist ein Zeiger auf einen Commit. Ein Commit in einem Worktree schiebt den Zeiger seines Branches weiter.
Wären zwei Worktrees auf demselben Branch, würde ein Commit im einen den Branch auch für den anderen weiterschieben.
Dessen Dateien passten dann nicht mehr zu seinem `HEAD`, und `git status` zeigte Änderungen, die dort niemand gemacht hat.

Deshalb gehört ein Branch zu genau einem Worktree, und jeder Agent bekommt seinen eigenen.
Mit `--force` geht es trotzdem; dann muss man selbst aufpassen.

## Zusatz: committen und zusammenführen

Die Änderung aus Schritt 3 committen. Dann in `handson-b` nachsehen: Kennt dieser Worktree den neuen Commit auf `demo/a`? Steht die Zeile schon in seiner `README.md`? Zum Schluss `demo/a` in `handson-b` mergen.

Hilfreich: `git commit -am "…"`, `git -C ../handson-b log --oneline -1 demo/a`, `git -C ../handson-b merge demo/a`.

In [8]:
%%bash
cd "$UEBUNG/handson-a"
git status --short
# hier weiter

 M README.md


In [9]:
# @title Lösung (aufklappen)
get_ipython().run_cell_magic("bash", "", """
cd "$UEBUNG/handson-a"
git diff --quiet || git commit --quiet -am "README im Worktree handson-a ergänzt"
git log --oneline -1

echo
echo "handson-b kennt den Commit, die Historie ist gemeinsam:"
git -C ../handson-b log --oneline -1 demo/a
echo "Die Datei in handson-b ist noch die alte:"
tail -1 ../handson-b/README.md

echo
echo "Nach dem Merge in handson-b:"
git -C ../handson-b merge --quiet demo/a
tail -1 ../handson-b/README.md
""")

94a301f README im Worktree handson-a ergänzt



handson-b kennt den Commit, die Historie ist gemeinsam:


94a301f README im Worktree handson-a ergänzt


Die Datei in handson-b ist noch die alte:


  von Jeremy Howard.



Nach dem Merge in handson-b:


Geändert im Worktree handson-a


## 5. Aufräumen

`git worktree remove` löscht den Ordner, der Branch bleibt. Einen Worktree mit Änderungen, die nicht committet sind, löscht Git nur mit `--force`. Ist die Änderung aus Schritt 3 noch nicht committet, meldet Git das bei `handson-a`, und die Zelle löscht ihn mit `--force`. Zum Schluss verschwindet der ganze Übungsordner.

In [10]:
%%bash
cd "$UEBUNG/handson"
git worktree remove ../handson-b
git worktree remove ../handson-a || git worktree remove --force ../handson-a
git worktree list
git branch --list 'demo/*'
rm -rf "$UEBUNG"

/private/var/folders/n0/3rmmylfj6xg_f6lfznjt8t380000gn/T/uebung-worktrees/handson 31838f4 [main]


  demo/a
  demo/b
